# 04 · Subgroups and dataset shift

In [1]:
import json

import matplotlib.pyplot as plt
import pandas as pd

from medpredict.common.config import get_settings
from medpredict.storage.database import Database

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)
db = Database(get_settings().db_path)
sg = db.query_df("SELECT grouping, \"group\", n, n_positive, suppressed, auroc, auroc_lo, auroc_hi, sensitivity, mean_predicted, observed_rate FROM subgroups")
sg.round(3)

,grouping,group,n,n_positive,suppressed,auroc,auroc_lo,auroc_hi,sensitivity,mean_predicted,observed_rate
0,sex,F,629,207,0,0.671,0.624,0.713,0.802,0.308,0.329
1,sex,M,522,146,0,0.683,0.635,0.733,0.774,0.290,0.280
2,age_band,18-44,99,24,0,0.624,0.517,0.727,0.167,0.126,0.242
3,age_band,45-64,415,83,0,0.594,0.521,0.659,0.530,0.225,0.200
4,age_band,65-79,427,144,0,0.624,0.571,0.668,0.903,0.332,0.337
5,age_band,80+,210,102,0,0.708,0.636,0.773,0.990,0.463,0.486
6,site,A,518,125,0,0.674,0.631,0.729,0.776,0.283,0.241
7,site,B,408,127,0,0.674,0.614,0.723,0.772,0.286,0.311
8,site,C,225,101,0,0.646,0.579,0.723,0.832,0.362,0.449


In [2]:
db.query_df("SELECT test, level, auroc, brier, ece, mean_predicted, observed_rate FROM shift_results").round(3)

,test,level,auroc,brier,ece,mean_predicted,observed_rate
0,missingness,0%,0.678,0.194,0.029,0.300,0.307
1,missingness,10%,0.676,0.194,0.022,0.305,0.307
2,missingness,20%,0.679,0.193,0.029,0.311,0.307
3,missingness,30%,0.674,0.193,0.027,0.315,0.307
4,missingness,40%,0.672,0.194,0.026,0.321,0.307
5,covariate,older_population,0.680,0.195,0.049,0.349,0.307
6,covariate,higher_glucose,0.678,0.194,0.034,0.310,0.307
7,site,in-distribution sites,0.677,0.183,0.038,0.250,0.272
8,site,held-out site C,0.658,0.251,0.161,0.299,0.449
9,temporal,validation period,0.701,0.187,0.000,0.304,0.304


**Findings.**
* Discrimination is fairly stable under missingness and covariate shift, but **calibration fails on the
  held-out site** (risk under-estimated where prevalence is higher) — local recalibration is required.
* At one shared threshold, sensitivity differs strongly by age band; younger patients are rarely flagged.
  Small groups have wide CIs and are not over-interpreted.
* The validation-period ECE is ≈0 because the calibrator was fitted on that period (in-sample).